# RL Test Flow Optimization - GPU Training Pipeline

**Runtime**: Kaggle Kernel with T4 GPU (16GB VRAM)  
**Framework**: Stable-Baselines3 + sb3-contrib + Optuna  
**Author**: Rajendar Muddasani  

---

## Problem Statement

In semiconductor post-silicon validation, chips must pass through a sequence of tests
(voltage, timing, thermal, functional, etc.). Running all tests is expensive and time-consuming.
An RL agent learns the **optimal test ordering** to maximize defect detection while minimizing
cost and time.

## Approach

| Stage | Description | Steps | Time |
|-------|-------------|-------|------|
| 1 | Install + GPU check | - | ~2 min |
| 2 | Generate 100K-chip x 200-test dataset | - | ~3 min |
| 3 | Environment validation + sanity check | - | ~1 min |
| 4 | Baseline evaluation (random, greedy, cost-efficient) | 500 ep each | ~2 min |
| 5 | Stage-1: Train all 4 algorithms | 200K steps each | ~20 min |
| 6 | Stage-2: Deep train top-2 (3 seeds) | 500K steps each | ~30 min |
| 7 | Optuna HPO on best algorithm | 30 trials x 100K | ~45 min |
| 8 | Final training with best hyperparams | 1M steps | ~15 min |
| 9 | Curriculum learning (10/50/200 tests) | 200K each | ~10 min |
| 10-13 | Plots + artifact export | - | ~5 min |

## Algorithms Compared

| Algorithm | Type | Key Feature | Best For |
|-----------|------|-------------|----------|
| **MaskablePPO** | On-policy | Action masking support | Variable valid actions |
| **PPO** | On-policy | Clipped surrogate | General RL tasks |
| **DQN** | Off-policy | Experience replay | Discrete actions |
| **A2C** | On-policy | Synchronous advantage | Fast convergence |

### Defect Taxonomy (14 types, 5 groups)

| Group | Defect Types |
|-------|-------------|
| Electrical | voltage_droop, current_leak, esd_damage |
| Timing | setup_violation, hold_violation, clock_jitter, frequency_fail |
| Thermal | power_thermal, electromigration |
| Functional | logic_fail, memory_fail, io_fail, scan_fail |
| Analog | analog_drift |

> **GPU Requirements**: Kaggle T4 with 16GB VRAM. Total runtime ~2-3 hours.  
> **Production scale**: 1M chips x 1000 tests, 50+ Optuna trials on A100/H100.

## Step 1: Environment Setup

Install Stable-Baselines3 (SB3) and sb3-contrib for reinforcement learning,
Optuna for hyperparameter optimization, and MLflow for experiment tracking.

**Why these packages?**
- `stable-baselines3`: Production-quality implementations of PPO, DQN, A2C
- `sb3-contrib`: MaskablePPO - critical for action masking (not all tests valid at all times)
- `optuna`: Bayesian hyperparameter optimization (smarter than grid/random search)
- `mlflow`: Experiment tracking across all training runs
- `pyarrow`: Fast Parquet I/O for large test result datasets

In [ ]:
# Install RL and optimization dependencies
!pip install -q stable-baselines3[extra] sb3-contrib gymnasium optuna mlflow pyarrow matplotlib seaborn pandas numpy torch

# Always do a fresh clone to avoid stale cached files from previous Kaggle runs
import os, sys
!rm -rf rl-test-flow-optimization
!git clone https://github.com/rajendarmuddasani/rl-test-flow-optimization.git

os.chdir('rl-test-flow-optimization')
sys.path.insert(0, '.')

import torch
print(f'PyTorch:        {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:            {gpu_name}')
    print(f'VRAM:           {vram_gb:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be slow.')
    print('Go to Settings > Accelerator > GPU T4 x2')

import stable_baselines3 as sb3
import sb3_contrib
import optuna
print(f'SB3:            {sb3.__version__}')
print(f'sb3-contrib:    {sb3_contrib.__version__}')
print(f'Optuna:         {optuna.__version__}')


## Step 2: Generate Test Results Dataset

Generate a synthetic dataset simulating semiconductor test results for 100K chips
across 200 tests. Each chip:
- Has a 70% defect rate (realistic for validation batches)
- Is assigned a defect type from the 14-type taxonomy
- Gets pass/fail results for each test (based on test sensitivity to defect type)

The data is saved in Parquet format for fast loading. Each test has properties:
- **Cost**: How expensive to run (in arbitrary cost units)
- **Time**: How long the test takes (in minutes)
- **Defect coverage**: Probability of catching a defect
- **Category**: Which defect group the test targets

**Scaling note**: We use 100K chips x 200 tests for Kaggle T4.
Production would use 1M chips x 1000 tests on A100.

In [ ]:
from generate_test_data import generate_dataset
import json

# Generate 100K chips x 200 tests (Kaggle scale)
# Production: n_chips=1_000_000, n_tests=1000
summary = generate_dataset(
    output_dir='data',
    n_chips=100_000,      # 100K chips to test
    n_tests=200,          # 200 available tests
    defect_rate=0.70,     # 70% of chips have defects
    seed=42,              # Reproducible results
    chunk_size=25_000,    # Process in 25K-chip chunks (memory efficient)
    fmt='parquet',        # Fast columnar format
)

print('Dataset Summary:')
print(json.dumps(summary, indent=2))

## Step 3: Environment Validation

Before training, we verify our Gymnasium environment works correctly:

1. **Observation space**: Each observation has `5 * n_tests + 3` dimensions
   - Per test (5 features): cost, time, coverage, already_run, estimated_value
   - Global (3 features): budget_remaining_pct, time_remaining_pct, tests_run_pct

2. **Action space**: `n_tests + 1` discrete actions (select test 0..n-1, or STOP)

3. **Action masking**: Already-executed tests and unaffordable tests are masked out.
   This is critical for MaskablePPO - it prevents the agent from selecting invalid actions.

4. **Reward design**: +1 for catching a defect, -0.1 per test cost, +5 bonus for full coverage

We test at three scales (10, 50, 200 tests) to verify the environment scales correctly.

In [ ]:
import numpy as np
from src.environment import TestFlowEnv, DEFECT_CATEGORIES, CATEGORY_GROUPS

print('Environment Validation')
print('=' * 60)

# Test at different scales
for n in [10, 50, 200]:
    env = TestFlowEnv(n_tests=n, cost_budget=50.0, time_budget=120.0)
    obs, _ = env.reset(seed=42)
    masks = env.action_masks()

    print(f'\nn_tests={n:4d}:')
    print(f'  Observation dim:  {obs.shape[0]:5d} (expected: {5*n+3})')
    print(f'  Action space:     {env.action_space.n:5d} (expected: {n+1})')
    print(f'  Valid actions:    {int(masks.sum()):5d}')
    print(f'  Mask dtype:       {masks.dtype}')

# Run a complete episode with random valid actions
print(f'\n{"=" * 60}')
print('Random Episode Walkthrough (200 tests)')
print('=' * 60)

env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)
obs, _ = env.reset(seed=42)
done, total_reward, steps = False, 0.0, 0

while not done:
    mask = env.action_masks()
    valid_actions = np.where(mask)[0]
    action = np.random.choice(valid_actions)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    steps += 1
    done = terminated or truncated

    # Print first 5 steps for illustration
    if steps <= 5:
        print(f'  Step {steps}: action={action:3d}, reward={reward:+.3f}, '
              f'valid_remaining={int(env.action_masks().sum())}')

print(f'  ...')
print(f'  Episode done after {steps} steps')
print(f'  Total reward:  {total_reward:+.2f}')
print(f'  Info: {info}')

print(f'\nDefect categories: {len(DEFECT_CATEGORIES)}')
print(f'Category groups:   {list(CATEGORY_GROUPS.keys())}')

## Step 4: Baseline Evaluation

Before training RL agents, we establish baselines using simple heuristic policies.
These serve as lower bounds - our RL agents must beat these to justify the complexity.

| Baseline | Strategy | Expected Performance |
|----------|----------|---------------------|
| **Random** | Randomly pick from valid tests | Poor - no intelligence |
| **Greedy Coverage** | Always pick highest-coverage test | Good accuracy, high cost |
| **Cost Efficient** | Best coverage/cost ratio, stop at 40% budget | Lower cost, may miss defects |

We evaluate each baseline over 500 episodes for statistical significance.

**Key metrics:**
- **Mean reward**: Overall optimization objective
- **Accuracy**: Fraction of defective chips correctly identified
- **Mean cost**: Average testing cost per chip
- **Mean tests**: Average number of tests executed per chip

In [ ]:
from src.agent import BASELINES, evaluate_policy
import pandas as pd

env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)

print('Baseline Evaluation (500 episodes each)')
print('=' * 70)
print(f'{"Policy":20s} {"Reward":>10s} {"Accuracy":>10s} {"Cost":>10s} {"Tests":>10s}')
print('-' * 70)

baseline_results = {}
for name, policy_fn in BASELINES.items():
    metrics = evaluate_policy(env, policy_fn, n_episodes=500)
    baseline_results[name] = metrics
    print(f'{name:20s} {metrics["mean_reward"]:>+10.2f} {metrics["accuracy"]:>10.3f} '
          f'{metrics["mean_cost"]:>10.2f} {metrics["mean_tests"]:>10.1f}')

print('=' * 70)

baseline_df = pd.DataFrame(baseline_results).T
print(f'\nBest baseline: {baseline_df["mean_reward"].idxmax()} '
      f'(reward={baseline_df["mean_reward"].max():+.2f})')
print(f'\nRL agents must beat: {baseline_df["mean_reward"].max():+.2f} reward')

## Step 5: Stage-1 - Train All 4 Algorithms (200K steps)

The first training stage runs all 4 algorithms with default hyperparameters
for 200K steps each. This is a **screening round** to identify which algorithms
work best for our specific problem.

**Why 200K steps?** It is enough for the agent to learn basic policies - pick high-coverage
tests early, avoid expensive tests when budget is low, and learn when to STOP.
Not enough for convergence, but sufficient to rank algorithms.

**Training details per algorithm:**
- All use MlpPolicy (2-layer MLP with 64 hidden units)
- EvalCallback evaluates every 5K steps, saves best model
- Discount factor gamma=0.99 (future rewards matter)
- Action masking is only used by MaskablePPO; others must learn to avoid invalid actions

In [ ]:
import time as _time
import pandas as pd
from src.agent import ALGO_REGISTRY, evaluate_trained_model

STAGE1_STEPS = 200_000
stage1_results = {}

print(f'Stage-1: Training 4 algorithms x {STAGE1_STEPS:,} steps')
print('=' * 70)

for algo_name, train_fn in ALGO_REGISTRY.items():
    print(f'\n>>> Training {algo_name.upper()}')
    env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)

    t0 = _time.time()
    model = train_fn(
        env,
        total_timesteps=STAGE1_STEPS,
        output_dir=f'outputs/stage1/{algo_name}',
    )
    train_time = _time.time() - t0

    # Evaluate trained model over 500 episodes
    metrics = evaluate_trained_model(env, model, n_episodes=500)
    metrics['train_time_sec'] = round(train_time, 1)
    stage1_results[algo_name] = metrics

    print(f'  {algo_name}: reward={metrics["mean_reward"]:+.2f} | '
          f'accuracy={metrics["accuracy"]:.3f} | '
          f'cost={metrics["mean_cost"]:.2f} | '
          f'time={train_time:.1f}s')

# Summary table
stage1_df = pd.DataFrame(stage1_results).T
print(f'\n{"=" * 70}')
print('Stage-1 Summary')
print(stage1_df.to_string())

# Compare to best baseline
best_bl = baseline_df['mean_reward'].max()
for algo, row in stage1_df.iterrows():
    improvement = row['mean_reward'] - best_bl
    pct = (improvement / abs(best_bl)) * 100 if best_bl != 0 else 0
    print(f'  {algo} vs best baseline: {improvement:+.2f} ({pct:+.1f}%)')

## Step 6: Stage-2 - Deep Training Top-2 Algorithms (500K steps, 3 seeds)

We take the **top-2 algorithms** from Stage-1 and train them with:
- **500K steps** (2.5x more than Stage-1) for better convergence
- **3 random seeds** to measure variance and ensure results are not lucky

**Why 3 seeds?** RL training is inherently stochastic. A single run might converge
to a local optimum by chance. Running 3 seeds gives us mean + standard deviation,
showing how reliable each algorithm is.

**Selection criteria**: We rank by mean_reward and pick the top 2.
If the top-2 are close (<5% difference), seed stability becomes the tiebreaker.

In [ ]:
# Select top-2 algorithms by reward
ranked = stage1_df.sort_values('mean_reward', ascending=False)
top2_algos = list(ranked.index[:2])
print(f'Top-2 algorithms from Stage-1: {top2_algos}')
print(f'  {top2_algos[0]}: {ranked.iloc[0]["mean_reward"]:+.2f}')
print(f'  {top2_algos[1]}: {ranked.iloc[1]["mean_reward"]:+.2f}')

STAGE2_STEPS = 500_000
SEEDS = [42, 123, 777]
stage2_results = {}

print(f'\nStage-2: {STAGE2_STEPS:,} steps x {len(SEEDS)} seeds')
print('=' * 70)

for algo_name in top2_algos:
    train_fn = ALGO_REGISTRY[algo_name]
    seed_metrics = []

    for seed in SEEDS:
        print(f'\n>>> {algo_name} seed={seed}')
        env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)

        t0 = _time.time()
        model = train_fn(
            env,
            total_timesteps=STAGE2_STEPS,
            output_dir=f'outputs/stage2/{algo_name}_seed{seed}',
            seed=seed,
        )
        train_time = _time.time() - t0

        metrics = evaluate_trained_model(env, model, n_episodes=500)
        metrics['seed'] = seed
        metrics['train_time_sec'] = round(train_time, 1)
        seed_metrics.append(metrics)

        print(f'  reward={metrics["mean_reward"]:+.2f} | accuracy={metrics["accuracy"]:.3f} | '
              f'cost={metrics["mean_cost"]:.2f} | time={train_time:.1f}s')

    stage2_results[algo_name] = seed_metrics

# Seed stability summary
print(f'\n{"=" * 70}')
print('Stage-2 Seed Stability')
print(f'{"Algorithm":20s} {"Mean Reward":>12s} {"Std Reward":>12s} {"Mean Acc":>10s}')
print('-' * 60)
for algo in top2_algos:
    rewards = [m['mean_reward'] for m in stage2_results[algo]]
    accs = [m['accuracy'] for m in stage2_results[algo]]
    print(f'{algo:20s} {np.mean(rewards):>+12.2f} {np.std(rewards):>12.2f} {np.mean(accs):>10.3f}')

## Step 7: Optuna Hyperparameter Optimization

With the best algorithm identified, we use **Optuna** to find optimal hyperparameters.
Optuna uses a Tree-structured Parzen Estimator (TPE) for Bayesian optimization,
which is much more efficient than grid search.

**Hyperparameters being optimized:**

| Parameter | Search Range | What It Controls |
|-----------|-------------|------------------|
| learning_rate | 1e-5 to 1e-2 (log) | How fast the agent updates weights |
| gamma | 0.9 to 0.999 | How much future rewards matter |
| batch_size | 32, 64, 128, 256 | Samples per gradient update |

Each trial trains for 100K steps (shorter than full training for speed),
but enough to differentiate good vs bad hyperparameters.

**30 trials x 100K steps** should take ~45 minutes on Kaggle T4.

In [ ]:
from src.agent import run_optuna_hpo

best_algo = top2_algos[0]
print(f'Running Optuna HPO for: {best_algo}')
print(f'  Trials: 30')
print(f'  Steps per trial: 100,000')
print(f'  Search space: learning_rate (1e-5..1e-2), gamma (0.9..0.999), batch_size (32..256)')
print()

hpo_result = run_optuna_hpo(
    env_cls=TestFlowEnv,
    env_kwargs={'n_tests': 200, 'cost_budget': 50.0, 'time_budget': 120.0},
    algo=best_algo,
    n_trials=30,
    timesteps=100_000,
)

print(f'\n{"=" * 50}')
print(f'HPO Complete')
print(f'{"=" * 50}')
print(f'Best reward:  {hpo_result["best_value"]:+.2f}')
print(f'Best params:')
for k, v in hpo_result['best_params'].items():
    print(f'  {k}: {v}')
print(f'Total trials: {hpo_result["n_trials"]}')

## Step 8: Final Training - Best Algorithm + Best Hyperparameters (1M steps)

Now we combine the **best algorithm** (from Stage-1/2) with the **best hyperparameters**
(from Optuna) and train for **1 million steps** - 5x more than Stage-1.

At 1M steps, the agent should:
- Have fully converged policy
- Learned optimal test ordering for each defect type
- Learned when to STOP testing (cost/benefit tradeoff)
- Generalized across different chip defect profiles

This is the model we will deploy for production inference.

In [ ]:
FINAL_STEPS = 1_000_000
best_params = hpo_result['best_params']
train_fn = ALGO_REGISTRY[best_algo]

env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)

print(f'Final Training')
print(f'  Algorithm: {best_algo}')
print(f'  Steps:     {FINAL_STEPS:,}')
print(f'  Params:    {best_params}')
print()

t0 = _time.time()
final_model = train_fn(
    env,
    total_timesteps=FINAL_STEPS,
    output_dir='outputs/final',
    seed=42,
    **best_params,
)
final_time = _time.time() - t0

# Evaluate final model over 1000 episodes (more than earlier for precision)
final_metrics = evaluate_trained_model(env, final_model, n_episodes=1000)

print(f'\n{"=" * 50}')
print(f'FINAL RESULTS ({final_time:.0f}s training)')
print(f'{"=" * 50}')
for k, v in final_metrics.items():
    print(f'  {k:15s}: {v:.4f}')

# Compare to best baseline
best_bl_reward = baseline_df['mean_reward'].max()
improvement = final_metrics['mean_reward'] - best_bl_reward
pct = (improvement / abs(best_bl_reward)) * 100 if best_bl_reward != 0 else 0
print(f'\nImprovement over best baseline: {improvement:+.2f} ({pct:+.1f}%)')

## Step 9: Curriculum Learning (10 -> 50 -> 200 tests)

**Curriculum learning** trains the agent on progressively harder problems:
1. Start with 10 tests (easy - few choices to make)
2. Scale to 50 tests (medium - more complex tradeoffs)
3. Full 200 tests (hard - must handle large action space)

This demonstrates that our environment and agent scale correctly.
In production, the curriculum would extend to 500 and 1000 tests.

**What we expect:**
- Reward should decrease as n_tests increases (harder problem)
- Accuracy should remain high (agent learns to find defects at any scale)
- Training time should increase proportionally

In [ ]:
curriculum_results = {}
CURRICULUM_STEPS = 200_000

print(f'Curriculum Learning: {CURRICULUM_STEPS:,} steps per scale')
print('=' * 60)

for n_tests in [10, 50, 200]:
    print(f'\n>>> n_tests={n_tests}')
    env = TestFlowEnv(n_tests=n_tests, cost_budget=50.0, time_budget=120.0)

    t0 = _time.time()
    model = train_fn(
        env,
        total_timesteps=CURRICULUM_STEPS,
        output_dir=f'outputs/curriculum/{n_tests}',
        seed=42,
        **best_params,
    )
    elapsed = _time.time() - t0

    metrics = evaluate_trained_model(env, model, n_episodes=500)
    metrics['train_time_sec'] = round(elapsed, 1)
    curriculum_results[n_tests] = metrics

    print(f'  reward={metrics["mean_reward"]:+.2f} | accuracy={metrics["accuracy"]:.3f} | '
          f'cost={metrics["mean_cost"]:.2f} | tests={metrics["mean_tests"]:.1f} | '
          f'time={elapsed:.1f}s')

# Summary table
import pandas as pd
curr_df = pd.DataFrame(curriculum_results).T
curr_df.index.name = 'n_tests'
print(f'\n{"=" * 60}')
print('Curriculum Summary')
print(curr_df.to_string())

## Step 10: Publication-Quality Plots (Main Results)

Generate a 2x3 panel figure showing the complete experimental results:

1. **Stage-1 Algorithm Comparison**: Which algorithm learns best in 200K steps?
2. **Detection Accuracy**: Baselines vs RL agents head-to-head
3. **Cost vs Accuracy Tradeoff**: Pareto frontier of all methods
4. **Seed Stability**: How reliable is the top algorithm across random seeds?
5. **Curriculum Scaling**: How performance changes with problem complexity
6. **Final Model vs Baselines**: The money plot - does RL actually help?

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 150})
from pathlib import Path

Path('outputs').mkdir(exist_ok=True)

colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Panel 1: Stage-1 Algorithm Comparison ---
ax = axes[0, 0]
algos = list(stage1_df.index)
rewards = stage1_df['mean_reward'].values
ax.bar(algos, rewards, color=colors[:len(algos)])
ax.set_title('Stage-1: Algorithm Comparison', fontweight='bold')
ax.set_ylabel('Mean Reward')
ax.axhline(y=baseline_df.loc['greedy_coverage', 'mean_reward'],
           color='gray', linestyle='--', label='Greedy Baseline')
ax.legend()
for i, v in enumerate(rewards):
    ax.text(i, v + 0.02 * abs(v), f'{v:.1f}', ha='center', fontsize=9)

# --- Panel 2: Accuracy Comparison (all methods) ---
ax = axes[0, 1]
all_labels = list(baseline_df.index) + list(stage1_df.index)
all_accs = list(baseline_df['accuracy'].values) + list(stage1_df['accuracy'].values)
bar_colors = ['#95a5a6'] * len(baseline_df) + colors[:len(stage1_df)]
ax.barh(all_labels, all_accs, color=bar_colors)
ax.set_title('Detection Accuracy: Baselines vs RL', fontweight='bold')
ax.set_xlim(0, 1)
for i, v in enumerate(all_accs):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

# --- Panel 3: Cost vs Accuracy Scatter ---
ax = axes[0, 2]
for name, row in baseline_df.iterrows():
    ax.scatter(row['mean_cost'], row['accuracy'], marker='x', s=100, color='gray', zorder=5)
    ax.annotate(name, (row['mean_cost'], row['accuracy']), fontsize=8, xytext=(5, 5),
                textcoords='offset points')
for i, (name, row) in enumerate(stage1_df.iterrows()):
    ax.scatter(row['mean_cost'], row['accuracy'], marker='o', s=120, color=colors[i], zorder=5)
    ax.annotate(name, (row['mean_cost'], row['accuracy']), fontsize=8, xytext=(5, 5),
                textcoords='offset points')
ax.set_xlabel('Mean Cost')
ax.set_ylabel('Accuracy')
ax.set_title('Cost vs Accuracy Trade-off', fontweight='bold')

# --- Panel 4: Seed Stability Boxplots ---
ax = axes[1, 0]
for i, algo in enumerate(top2_algos):
    seed_rewards = [m['mean_reward'] for m in stage2_results[algo]]
    bp = ax.boxplot([seed_rewards], positions=[i], widths=0.5, patch_artist=True)
    bp['boxes'][0].set_facecolor(colors[i])
ax.set_xticks(range(len(top2_algos)))
ax.set_xticklabels(top2_algos)
ax.set_title('Stage-2: Seed Stability (3 seeds)', fontweight='bold')
ax.set_ylabel('Mean Reward')

# --- Panel 5: Curriculum Scaling ---
ax = axes[1, 1]
n_vals = list(curriculum_results.keys())
cr_r = [curriculum_results[n]['mean_reward'] for n in n_vals]
cr_a = [curriculum_results[n]['accuracy'] for n in n_vals]
ax.plot(n_vals, cr_r, 'o-', color='#2ecc71', label='Reward', linewidth=2, markersize=8)
ax2 = ax.twinx()
ax2.plot(n_vals, cr_a, 's--', color='#e74c3c', label='Accuracy', linewidth=2, markersize=8)
ax.set_xlabel('Number of Tests')
ax.set_ylabel('Reward', color='#2ecc71')
ax2.set_ylabel('Accuracy', color='#e74c3c')
ax.set_title('Curriculum Scaling', fontweight='bold')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

# --- Panel 6: Final Model vs Baselines ---
ax = axes[1, 2]
labels = list(baseline_df.index) + [f'{best_algo}\n(1M steps)']
rews = list(baseline_df['mean_reward'].values) + [final_metrics['mean_reward']]
bc = ['#95a5a6'] * len(baseline_df) + ['#2ecc71']
ax.bar(labels, rews, color=bc)
ax.set_title('Final Model vs Baselines', fontweight='bold')
ax.set_ylabel('Mean Reward')
for i, v in enumerate(rews):
    ax.text(i, v + 0.02 * abs(v), f'{v:.1f}', ha='center', fontsize=9)

plt.suptitle('RL Test Flow Optimization - Complete Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/training_results.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/training_results.png')

## Step 11: Detailed Comparison - Final Model vs Greedy Baseline

Deep dive into how the RL agent's behavior differs from the greedy baseline:
- **Reward distribution**: Is the RL agent consistently better, or just on average?
- **Tests run distribution**: Does the RL agent run fewer tests (more efficient)?
- **Cost distribution**: How much cheaper is the RL approach?

These histograms reveal whether the improvement is uniform or concentrated
on specific chip types.

In [ ]:
from src.agent import BASELINES

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
env = TestFlowEnv(n_tests=200, cost_budget=50.0, time_budget=120.0)

def collect_episodes(policy, n=500):
    """Run episodes and collect per-episode metrics."""
    rewards, tests_run, costs = [], [], []
    for _ in range(n):
        obs, _ = env.reset()
        done, ep_reward = False, 0.0
        while not done:
            if callable(policy):
                action = policy(env)
            else:
                action, _ = policy.predict(obs, deterministic=True)
            obs, r, t, tr, info = env.step(action)
            ep_reward += r
            done = t or tr
        rewards.append(ep_reward)
        tests_run.append(info.get('tests_run', 0))
        costs.append(info.get('cost', 0))
    return rewards, tests_run, costs

# Collect data for both policies
print('Collecting 500 episodes each for greedy and final model...')
r_g, t_g, c_g = collect_episodes(BASELINES['greedy_coverage'])
r_f, t_f, c_f = collect_episodes(final_model)

# Plot distributions
axes[0].hist(r_g, bins=30, alpha=0.6, label='Greedy', color='gray')
axes[0].hist(r_f, bins=30, alpha=0.6, label=f'{best_algo} (Final)', color='#2ecc71')
axes[0].set_xlabel('Episode Reward')
axes[0].set_title('Reward Distribution', fontweight='bold')
axes[0].legend()
axes[0].axvline(np.mean(r_g), color='gray', linestyle='--', alpha=0.7)
axes[0].axvline(np.mean(r_f), color='#2ecc71', linestyle='--', alpha=0.7)

axes[1].hist(t_g, bins=20, alpha=0.6, label='Greedy', color='gray')
axes[1].hist(t_f, bins=20, alpha=0.6, label=f'{best_algo} (Final)', color='#3498db')
axes[1].set_xlabel('Tests Run per Chip')
axes[1].set_title('Test Efficiency', fontweight='bold')
axes[1].legend()

axes[2].hist(c_g, bins=20, alpha=0.6, label='Greedy', color='gray')
axes[2].hist(c_f, bins=20, alpha=0.6, label=f'{best_algo} (Final)', color='#e74c3c')
axes[2].set_xlabel('Total Cost per Chip')
axes[2].set_title('Cost Distribution', fontweight='bold')
axes[2].legend()

plt.suptitle(f'Detailed: {best_algo} vs Greedy Baseline (500 episodes each)', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/detailed_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'\nMean tests: Greedy={np.mean(t_g):.1f}, {best_algo}={np.mean(t_f):.1f} '
      f'({(np.mean(t_f) - np.mean(t_g)) / np.mean(t_g) * 100:+.1f}%)')
print(f'Mean cost:  Greedy={np.mean(c_g):.2f}, {best_algo}={np.mean(c_f):.2f} '
      f'({(np.mean(c_f) - np.mean(c_g)) / np.mean(c_g) * 100:+.1f}%)')

## Step 12: Test Selection Heatmap + Final Summary Table

**Left**: Heatmap showing which tests the trained agent selects most frequently.
High-frequency tests are the agent's "go-to" choices - typically high-coverage,
low-cost tests that efficiently detect defects.

**Right**: Summary comparison table of key metrics between greedy baseline
and the final trained model.

The heatmap reveals the agent's learned strategy - does it focus on a few
reliable tests, or spread selections across many?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Test selection frequency heatmap
test_freq = np.zeros(min(50, env.n_tests))  # Track first 50 tests
for _ in range(500):
    obs, _ = env.reset()
    done = False
    while not done:
        action, _ = final_model.predict(obs, deterministic=True)
        obs, r, t, tr, info = env.step(action)
        done = t or tr
        if action < len(test_freq):
            test_freq[action] += 1

# Reshape into grid for heatmap
cols = 10
rows_hm = (len(test_freq) + cols - 1) // cols
padded = np.zeros(rows_hm * cols)
padded[:len(test_freq)] = test_freq
im = axes[0].imshow(padded.reshape(rows_hm, cols), cmap='YlOrRd', aspect='auto')
axes[0].set_title('Test Selection Frequency (first 50 tests)', fontweight='bold')
axes[0].set_xlabel('Test ID (mod 10)')
axes[0].set_ylabel('Test ID (div 10)')
plt.colorbar(im, ax=axes[0], label='Selection Count (500 episodes)')

# Right: Summary table
axes[1].axis('off')
tbl_data = [
    ['Reward', f'{baseline_df.loc["greedy_coverage", "mean_reward"]:.2f}',
     f'{final_metrics["mean_reward"]:.2f}'],
    ['Accuracy', f'{baseline_df.loc["greedy_coverage", "accuracy"]:.3f}',
     f'{final_metrics["accuracy"]:.3f}'],
    ['Cost', f'{baseline_df.loc["greedy_coverage", "mean_cost"]:.2f}',
     f'{final_metrics["mean_cost"]:.2f}'],
    ['Tests/Chip', f'{baseline_df.loc["greedy_coverage", "mean_tests"]:.1f}',
     f'{final_metrics["mean_tests"]:.1f}'],
]
table = axes[1].table(
    cellText=tbl_data,
    colLabels=['Metric', 'Greedy Baseline', f'{best_algo} (Final)'],
    loc='center', cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(13)
table.scale(1, 2.2)

# Color the winning cells
for i in range(len(tbl_data)):
    # Color header row
    table[(0, 0)].set_facecolor('#e8e8e8')
    table[(0, 1)].set_facecolor('#e8e8e8')
    table[(0, 2)].set_facecolor('#e8e8e8')

axes[1].set_title('Final Summary: Greedy vs RL Agent', fontweight='bold', y=0.88)

plt.tight_layout()
plt.savefig('outputs/heatmap_summary.png', bbox_inches='tight', dpi=150)
plt.show()

## Step 13: Export All Artifacts

Collect all training artifacts into a single directory for download:
- **Trained model weights** (.zip files for each algorithm)
- **Metrics JSON** with all results (baselines, stage-1, stage-2, HPO, final)
- **Plot images** for README and portfolio

**After this cell:** Download `outputs/artifacts/` and copy to your local repo:
1. `best_model.zip` -> `models/` directory
2. `*.png` files -> `assets/` directory for README
3. `all_metrics.json` -> `results/` directory

Then update the README with actual metric values before pushing to GitHub.

In [ ]:
import shutil
import json
from pathlib import Path

artifact_dir = Path('outputs/artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)

# Save all metrics as JSON
all_metrics = {
    'baselines': baseline_results,
    'stage1': stage1_results,
    'stage2': {k: v for k, v in stage2_results.items()},
    'hpo': hpo_result,
    'final': final_metrics,
    'curriculum': {str(k): v for k, v in curriculum_results.items()},
    'best_algo': best_algo,
    'best_params': hpo_result['best_params'],
}

with open(artifact_dir / 'all_metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

# Copy best model
best_model_path = Path(f'outputs/final/{best_algo}.zip')
if best_model_path.exists():
    shutil.copy(best_model_path, artifact_dir / 'best_model.zip')
    print(f'Copied best model: {best_model_path.stat().st_size / 1e6:.2f} MB')

# Copy all plots
for png in Path('outputs').glob('*.png'):
    shutil.copy(png, artifact_dir / png.name)

# Print inventory
print(f'\n{"=" * 50}')
print('ARTIFACT INVENTORY')
print(f'{"=" * 50}')
total_size = 0
for f in sorted(artifact_dir.iterdir()):
    size = f.stat().st_size
    total_size += size
    unit = 'KB' if size < 1e6 else 'MB'
    val = size / 1024 if size < 1e6 else size / 1e6
    print(f'  {f.name:35s} {val:6.1f} {unit}')
print(f'{"=" * 50}')
print(f'Total: {total_size / 1e6:.1f} MB')

# Final summary
print(f'\n{"=" * 50}')
print('FINAL RESULTS SUMMARY')
print(f'{"=" * 50}')
print(f'  Best algorithm:     {best_algo}')
print(f'  Best hyperparams:   {hpo_result["best_params"]}')
print(f'  Final reward:       {final_metrics["mean_reward"]:+.2f}')
print(f'  Final accuracy:     {final_metrics["accuracy"]:.3f}')
print(f'  Final cost:         {final_metrics["mean_cost"]:.2f}')
print(f'  Final tests/chip:   {final_metrics["mean_tests"]:.1f}')
print(f'  Baseline reward:    {baseline_df["mean_reward"].max():+.2f}')
print(f'  Improvement:        {final_metrics["mean_reward"] - baseline_df["mean_reward"].max():+.2f}')
print(f'\nDownload outputs/artifacts/ for your local repo.')